# Home Credit Default Risk - Feature Engineering Notebook

## Objective
This notebook performs **report-driven feature engineering** using:

1. The structural audit report (`9.master_consolidated_report.csv`)
2. The processed train dataset (`final_train_before_eda.csv`)
3. The processed test dataset (`final_test_before_eda.csv`)

## Design Principles
- Use the structural audit report as the main decision guide
- Keep the pipeline safe and reproducible
- Handle missing values differently for **structural history features** vs **true missing features**
- Create practical domain features instead of generating unnecessary noise
- Keep train/test columns aligned at the end
- Save outputs into **Processed_2**

## Main Outputs
- `final_train_after_feature_engineering.csv`
- `final_test_after_feature_engineering.csv`
- `feature_engineering_summary.csv`
- `feature_engineering_metadata.json`

## 1. Setup

In [1]:
# Standard library imports
import os
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Data manipulation
import numpy as np
import pandas as pd

# Visualization (optional quick diagnostics)
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 250)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## 2. Configuration
Update the paths only if your local paths are different.

In [2]:
# ===============================
# FILE PATH CONFIGURATION
# ===============================

REPORT_PATH = r"C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\adfa\9.master_consolidated_report.csv"

TRAIN_FILE_PATH = r"C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Data\Processed\final_train_before_eda.csv"

TEST_FILE_PATH = r"C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Data\Processed\final_test_before_eda.csv"

SAVE_DIR = r"C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Data\Processed_2"

TARGET_COL = "TARGET"
ID_COL = "SK_ID_CURR"

# Engineering controls
MISSING_INDICATOR_THRESHOLD = 5.0          # create missing indicator if missing % >= threshold
EXTREME_MISSING_DROP_THRESHOLD = 95.0      # drop non-structural columns if missing % >= threshold
MAX_LOG_FEATURES = 60                      # max number of auto-generated log features
MAX_BIN_FEATURES = 20                      # max number of auto-generated binned features
LOW_CARDINALITY_THRESHOLD = 12             # one-hot encode if categorical unique count <= threshold
HIGH_CARDINALITY_THRESHOLD = 20            # frequency encode if categorical unique count > threshold

os.makedirs(SAVE_DIR, exist_ok=True)

In [3]:
# Validate paths
for name, path in {
    "REPORT_PATH": REPORT_PATH,
    "TRAIN_FILE_PATH": TRAIN_FILE_PATH,
    "TEST_FILE_PATH": TEST_FILE_PATH,
    "SAVE_DIR": SAVE_DIR
}.items():
    print(f"{name}: {path}")
    if name != "SAVE_DIR":
        print("  Exists ->", os.path.exists(path))
    else:
        print("  Exists ->", os.path.exists(path))

REPORT_PATH: C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\adfa\9.master_consolidated_report.csv
  Exists -> True
TRAIN_FILE_PATH: C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Data\Processed\final_train_before_eda.csv
  Exists -> True
TEST_FILE_PATH: C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Data\Processed\final_test_before_eda.csv
  Exists -> True
SAVE_DIR: C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Data\Processed_2
  Exists -> True


## 3. Load Data

In [4]:
# Load the structural audit report and train/test datasets
report_df = pd.read_csv(REPORT_PATH)
train_df = pd.read_csv(TRAIN_FILE_PATH)
test_df = pd.read_csv(TEST_FILE_PATH)

print("Report shape:", report_df.shape)
print("Train shape :", train_df.shape)
print("Test shape  :", test_df.shape)

Report shape: (584, 196)
Train shape : (307511, 584)
Test shape  : (48744, 583)


In [5]:
# Quick preview
display(report_df.head(3))
display(train_df.head(3))
display(test_df.head(3))

,Unnamed: 0,column,datatype_current_dtype_report,datatype_suggested_dtype_report,semantic_current_dtype_report,semantic_semantic_type_guess,key_id_like_flag,key_is_primary_key_like,key_is_foreign_key_like,pattern_current_dtype_report,pattern_boolean_like_string_detection,pattern_numeric_like_object_detection,pattern_date_like_object_detection,train_content_current_dtype_report,train_content_non_null_count,train_content_unique_count,train_content_unique_ratio,train_content_avg_string_length,train_content_text_vs_categorical_split,train_content_mixed_type_flag,clean_current_dtype_report,clean_special_character_flag,clean_whitespace_issue_flag,clean_format_inconsistency_flag,clean_leading_or_trailing_space_ratio,clean_multiple_space_ratio,clean_special_character_ratio,dtype,missing_count,missing_percentage,missing_group,imputation_strategy,recommendation,unique_values,non_missing_count,completeness,sample_values,dtype_r1,missing_count_r1,missing_percentage_r1,non_missing_count_r1,target_type,test_used,effect_size,p_value,signal_score,missing_indicator_recommended,missing_pattern_guess,positive_label,target_positive_rate_missing,target_positive_rate_non_missing,target_rate_gap_abs,odds_ratio,distribution_shift_jsd,direction,mutual_info,missing_count_r2,co_missing_columns_count,co_missing_columns,null_placeholder_found,detected_placeholders,overall_missingness_score,non_null_count_x,unique_count,unique_percentage,duplicate_count,duplicate_value_ratio,constant_flag,top_value_percentage,quasi_constant_flag,top_value,top_value_count,top_values_preview,non_null_count_y,dominant_value,dominant_value_pct,second_dominant_value,second_dominant_pct,top_values,suspicious_uniformity_flag,dominance_note,non_null_count,missing_count_x,missing_pct,cardinality_count,cardinality_group,high_cardinality_flag,very_high_cardinality_flag,potential_id_flag,numeric_categorical_flag,encoding_recommendation,missing_count_y,top_category,top_category_percentage,rare_category_count,rare_category_percentage,rare_grouping_recommended,category_distribution_balance,normalized_entropy,high_cardinality,unseen_category_risk,unseen_category_count,unseen_category_ratio,unseen_categories,potential_typos,train_category_count,test_category_count,train_top_categories,test_top_categories,count,min_value,max_value,range_value,mean_value,median_value,std_dev,coef_of_variation,p25,p75,iqr_x,skewness_x,kurtosis_x,zero_count,negative_count,q1_x,q3_x,iqr_y,lower_bound,upper_bound,outlier_count_x,outlier_percentage,missing_rate,mean,median,std,min,q1_y,q3_y,max,iqr,skewness_y,kurtosis_y,outlier_count_y,outlier_rate,skew_label,variance_flag,normality_pvalue,normality_label,iqr_q1,iqr_q3,iqr_value,iqr_lower_bound,iqr_upper_bound,iqr_lower_count,iqr_upper_count,iqr_total_count,iqr_total_pct,extreme_iqr_count,extreme_iqr_pct,zscore_lower_bound,zscore_upper_bound,zscore_lower_count,zscore_upper_count,zscore_total_count,zscore_total_pct,mod_zscore_mad,mod_zscore_count,mod_zscore_pct,pctl_lower_bound,pctl_upper_bound,pctl_count,pctl_pct,kurtosis,distribution_type_x,is_heavy_tailed,is_skewed,heavy_tail_flag,has_outliers,outlier_severity,suggested_treatment,winsorize_lower,winsorize_upper,distribution_type_y,transformation_recommendation,scaling_needed,clipping_recommendation,winsorization_recommendation,is_numeric,is_discrete,is_near_constant,zero_inflation_flag,binning_opportunity,structure_score,recommended_action,reason,zero_ratio
0,0,AMT_ANNUITY,float64,float64,float64,numeric_continuous,False,False,False,float64,False,False,False,float64,307499,13672,0.044462,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN,float64,12,0.000000,low_missing,mean/median,Mean (if normal) or Median (if skewed),13672,307499,100.000000,"['24700.5', '35698.5', '6750.0']",float64,12.000000,0.000000,"307,499.000000",binary_classification,fisher_exact,0.001851,0.616206,0.203939,False,likely_random_or_weak_signal,1.000000,0.000000,0.080732,0.080732,0.000000,0.203939,lower_positive_rate_when_missin

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,DAYS_EMPLOYED_ANOM,APP_CREDIT_INCOME_RATIO,APP_ANNUITY_INCOME_RATIO,...,INS_PAY_PERC_MAX,INS_PAY_PERC_MEAN,INS_PAY_PERC_VAR,INS_PAY_DIFF_MAX,INS_PAY_DIFF_MEAN,INS_PAY_DIFF_SUM,INS_PAY_DIFF_VAR,INS_DPD_MAX,INS_DPD_MEAN,INS_DPD_SUM,INS_DBD_MAX,INS_DBD_MEAN,INS_DBD_SUM,INS_LATE_PAYMENT_FLAG_MEAN,INS_LATE_PAYMENT_FLAG_SUM,INS_UNDERPAY_FLAG_MEAN,INS_UNDERPAY_FLAG_SUM,INS_LAST365_AMT_PAYMENT_MEAN,INS_LAST365_AMT_PAYMENT_SUM,INS_LAST365_PAY_PERC_MEAN,INS_LAST365_PAY_PERC_MAX,INS_LAST365_PAY_DIFF_MEAN,INS_LAST365_PAY_DIFF_SUM,INS_LAST365_DPD_MEAN,INS_LAST365_DPD_MAX,INS_LAST365_DPD_SUM,INS_LAST365_LATE_PAYMENT_FLAG_MEAN,INS_LAST365_LATE_PAYMENT_FLAG_SUM,POS_SK_ID_PREV_NUNIQUE,POS_MONTHS_BALANCE_MIN,POS_MONTHS_BALANCE_MAX,POS_MONTHS_BALANCE_SIZE,POS_CNT_INSTALMENT_MAX,POS_CNT_INSTALMENT_MEAN,POS_CNT_INSTALMENT_SUM,POS_CNT_INSTALMENT_FUTURE_MAX,POS_CNT_INSTALMENT_FUTURE_MEAN,POS_CNT_INSTALMENT_FUTURE_SUM,POS_SK_DPD_MAX,POS_SK_DPD_MEAN,POS_SK_DPD_SUM,POS_SK_DPD_DEF_MAX,POS_SK_DPD_DEF_MEAN,POS_SK_DPD_DEF_SUM,POS_POS_LATE_FLAG_MEAN,POS_POS_LATE_FLAG_SUM,POS_NAME_CONTRACT_STATUS_Active_MEAN,POS_NAME_CONTRACT_STATUS_Amortized debt_MEAN,POS_NAME_CONTRACT_STATUS_Approved_MEAN,POS_NAME_CONTRACT_STATUS_Canceled_MEAN,POS_NAME_CONTRACT_STATUS_Completed_MEAN,POS_NAME_CONTRACT_STATUS_Demand_MEAN,POS_NAME_CONTRACT_STATUS_Returned to the store_MEAN,POS_NAME_CONTRACT_STATUS_Signed_MEAN,POS_NAME_CONTRACT_STATUS_XNA_MEAN,POS_NAME_CONTRACT_STATUS_nan_MEAN,CC_SK_ID_PREV_NUNIQUE,CC_MONTHS_BALANCE_MIN,CC_MONTHS_BALANCE_MAX,CC_MONTHS_BALANCE_SIZE,CC_AMT_BALANCE_MAX,CC_AMT_BALANCE_MEAN,CC_AMT_BALANCE_SUM,CC_AMT_CREDIT_LIMIT_ACTUAL_MAX,CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN,CC_AMT_DRAWINGS_ATM_CURRENT_MAX,CC_AMT_DRAWINGS_ATM_CURRENT_MEAN,CC_AMT_DRAWINGS_ATM_CURRENT_SUM,CC_AMT_DRAWINGS_CURRENT_MAX,CC_AMT_DRAWI

,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,DAYS_EMPLOYED_ANOM,APP_CREDIT_INCOME_RATIO,APP_ANNUITY_INCOME_RATIO,APP_CREDIT_ANNUITY_RATIO,...,INS_PAY_PERC_MAX,INS_PAY_PERC_MEAN,INS_PAY_PERC_VAR,INS_PAY_DIFF_MAX,INS_PAY_DIFF_MEAN,INS_PAY_DIFF_SUM,INS_PAY_DIFF_VAR,INS_DPD_MAX,INS_DPD_MEAN,INS_DPD_SUM,INS_DBD_MAX,INS_DBD_MEAN,INS_DBD_SUM,INS_LATE_PAYMENT_FLAG_MEAN,INS_LATE_PAYMENT_FLAG_SUM,INS_UNDERPAY_FLAG_MEAN,INS_UNDERPAY_FLAG_SUM,INS_LAST365_AMT_PAYMENT_MEAN,INS_LAST365_AMT_PAYMENT_SUM,INS_LAST365_PAY_PERC_MEAN,INS_LAST365_PAY_PERC_MAX,INS_LAST365_PAY_DIFF_MEAN,INS_LAST365_PAY_DIFF_SUM,INS_LAST365_DPD_MEAN,INS_LAST365_DPD_MAX,INS_LAST365_DPD_SUM,INS_LAST365_LATE_PAYMENT_FLAG_MEAN,INS_LAST365_LATE_PAYMENT_FLAG_SUM,POS_SK_ID_PREV_NUNIQUE,POS_MONTHS_BALANCE_MIN,POS_MONTHS_BALANCE_MAX,POS_MONTHS_BALANCE_SIZE,POS_CNT_INSTALMENT_MAX,POS_CNT_INSTALMENT_MEAN,POS_CNT_INSTALMENT_SUM,POS_CNT_INSTALMENT_FUTURE_MAX,POS_CNT_INSTALMENT_FUTURE_MEAN,POS_CNT_INSTALMENT_FUTURE_SUM,POS_SK_DPD_MAX,POS_SK_DPD_MEAN,POS_SK_DPD_SUM,POS_SK_DPD_DEF_MAX,POS_SK_DPD_DEF_MEAN,POS_SK_DPD_DEF_SUM,POS_POS_LATE_FLAG_MEAN,POS_POS_LATE_FLAG_SUM,POS_NAME_CONTRACT_STATUS_Active_MEAN,POS_NAME_CONTRACT_STATUS_Amortized debt_MEAN,POS_NAME_CONTRACT_STATUS_Approved_MEAN,POS_NAME_CONTRACT_STATUS_Canceled_MEAN,POS_NAME_CONTRACT_STATUS_Completed_MEAN,POS_NAME_CONTRACT_STATUS_Demand_MEAN,POS_NAME_CONTRACT_STATUS_Returned to the store_MEAN,POS_NAME_CONTRACT_STATUS_Signed_MEAN,POS_NAME_CONTRACT_STATUS_XNA_MEAN,POS_NAME_CONTRACT_STATUS_nan_MEAN,CC_SK_ID_PREV_NUNIQUE,CC_MONTHS_BALANCE_MIN,CC_MONTHS_BALANCE_MAX,CC_MONTHS_BALANCE_SIZE,CC_AMT_BALANCE_MAX,CC_AMT_BALANCE_MEAN,CC_AMT_BALANCE_SUM,CC_AMT_CREDIT_LIMIT_ACTUAL_MAX,CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN,CC_AMT_DRAWINGS_ATM_CURRENT_MAX,CC_AMT_DRAWINGS_ATM_CURRENT_MEAN,CC_AMT_DRAWINGS_ATM_CURRENT_SUM,CC_AMT_DRAWINGS_CURREN

## 4. Basic Validation

In [6]:
print("TARGET present in train ->", TARGET_COL in train_df.columns)
print("TARGET present in test  ->", TARGET_COL in test_df.columns)
print("ID present in train     ->", ID_COL in train_df.columns)
print("ID present in test      ->", ID_COL in test_df.columns)

if TARGET_COL in train_df.columns:
    display(train_df[TARGET_COL].value_counts(dropna=False).rename("count").to_frame())
    display((train_df[TARGET_COL].value_counts(normalize=True, dropna=False) * 100).round(4).rename("pct").to_frame())

TARGET present in train -> True
TARGET present in test  -> False
ID present in train     -> True
ID present in test      -> True


,count
TARGET,
0,282686
1,24825


,pct
TARGET,
0,91.927100
1,8.072900


## 5. Helper Functions
These helper functions make the notebook robust even when some columns are missing.

In [7]:
def normalize_bool(series):
    """Convert mixed-type boolean flags from the report into clean boolean values."""
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True, "false": False,
            "1": True, "0": False,
            "yes": True, "no": False,
            "nan": False, "none": False
        })
        .fillna(False)
    )

def get_existing_cols(df, cols):
    """Return only columns that exist in a dataframe."""
    return [c for c in cols if c in df.columns]

def safe_divide(a, b):
    """Safe division that avoids division by zero and infinite values."""
    result = np.where((b == 0) | pd.isna(b), np.nan, a / b)
    return result

def ensure_same_columns(train_df, test_df):
    """Align train and test columns after encoding/feature creation."""
    train_aligned, test_aligned = train_df.align(test_df, join="outer", axis=1, fill_value=0)
    return train_aligned, test_aligned

def clip_with_bounds(series, lower=None, upper=None):
    """Clip a numeric series using optional lower and upper bounds."""
    s = series.copy()
    if lower is not None and pd.notna(lower):
        s = s.clip(lower=lower)
    if upper is not None and pd.notna(upper):
        s = s.clip(upper=upper)
    return s

def describe_df(df, name):
    print(f"{name} shape: {df.shape}")
    print(f"{name} numeric columns: {df.select_dtypes(include=[np.number]).shape[1]}")
    print(f"{name} categorical columns: {df.select_dtypes(exclude=[np.number]).shape[1]}")

def family_columns(df, prefixes):
    cols = []
    for prefix in prefixes:
        cols.extend([c for c in df.columns if c.startswith(prefix)])
    return sorted(set(cols))

def report_missing_col(report_df):
    for col in ["missing_pct", "missing_percentage"]:
        if col in report_df.columns:
            return col
    raise ValueError("Missing percentage column not found in report.")

def report_unique_col(report_df):
    for col in ["unique_count", "cardinality_count"]:
        if col in report_df.columns:
            return col
    raise ValueError("Unique count column not found in report.")

## 6. Standardize Structural Audit Report
This section prepares the report so it can drive engineering decisions.

In [8]:
# Keep a clean report copy
rpt = report_df.copy()

# Standardize key helper columns
rpt["column"] = rpt["column"].astype(str)

missing_col = report_missing_col(rpt)
unique_col = report_unique_col(rpt)

# Normalize commonly used flags if present
for flag_col in [
    "constant_flag", "quasi_constant_flag", "high_cardinality_flag",
    "very_high_cardinality_flag", "potential_id_flag", "numeric_categorical_flag",
    "is_numeric", "is_discrete", "is_near_constant", "zero_inflation_flag",
    "binning_opportunity", "is_skewed", "has_outliers", "heavy_tail_flag",
    "clipping_recommendation", "winsorization_recommendation"
]:
    if flag_col in rpt.columns:
        rpt[flag_col] = normalize_bool(rpt[flag_col])

# Make sure numeric report fields are numeric
numeric_report_cols = [
    missing_col, unique_col, "zero_ratio", "winsorize_lower", "winsorize_upper",
    "structure_score"
]
for col in numeric_report_cols:
    if col in rpt.columns:
        rpt[col] = pd.to_numeric(rpt[col], errors="coerce")

print("Standardized report ready.")
display(rpt.head(3))

Standardized report ready.


,Unnamed: 0,column,datatype_current_dtype_report,datatype_suggested_dtype_report,semantic_current_dtype_report,semantic_semantic_type_guess,key_id_like_flag,key_is_primary_key_like,key_is_foreign_key_like,pattern_current_dtype_report,pattern_boolean_like_string_detection,pattern_numeric_like_object_detection,pattern_date_like_object_detection,train_content_current_dtype_report,train_content_non_null_count,train_content_unique_count,train_content_unique_ratio,train_content_avg_string_length,train_content_text_vs_categorical_split,train_content_mixed_type_flag,clean_current_dtype_report,clean_special_character_flag,clean_whitespace_issue_flag,clean_format_inconsistency_flag,clean_leading_or_trailing_space_ratio,clean_multiple_space_ratio,clean_special_character_ratio,dtype,missing_count,missing_percentage,missing_group,imputation_strategy,recommendation,unique_values,non_missing_count,completeness,sample_values,dtype_r1,missing_count_r1,missing_percentage_r1,non_missing_count_r1,target_type,test_used,effect_size,p_value,signal_score,missing_indicator_recommended,missing_pattern_guess,positive_label,target_positive_rate_missing,target_positive_rate_non_missing,target_rate_gap_abs,odds_ratio,distribution_shift_jsd,direction,mutual_info,missing_count_r2,co_missing_columns_count,co_missing_columns,null_placeholder_found,detected_placeholders,overall_missingness_score,non_null_count_x,unique_count,unique_percentage,duplicate_count,duplicate_value_ratio,constant_flag,top_value_percentage,quasi_constant_flag,top_value,top_value_count,top_values_preview,non_null_count_y,dominant_value,dominant_value_pct,second_dominant_value,second_dominant_pct,top_values,suspicious_uniformity_flag,dominance_note,non_null_count,missing_count_x,missing_pct,cardinality_count,cardinality_group,high_cardinality_flag,very_high_cardinality_flag,potential_id_flag,numeric_categorical_flag,encoding_recommendation,missing_count_y,top_category,top_category_percentage,rare_category_count,rare_category_percentage,rare_grouping_recommended,category_distribution_balance,normalized_entropy,high_cardinality,unseen_category_risk,unseen_category_count,unseen_category_ratio,unseen_categories,potential_typos,train_category_count,test_category_count,train_top_categories,test_top_categories,count,min_value,max_value,range_value,mean_value,median_value,std_dev,coef_of_variation,p25,p75,iqr_x,skewness_x,kurtosis_x,zero_count,negative_count,q1_x,q3_x,iqr_y,lower_bound,upper_bound,outlier_count_x,outlier_percentage,missing_rate,mean,median,std,min,q1_y,q3_y,max,iqr,skewness_y,kurtosis_y,outlier_count_y,outlier_rate,skew_label,variance_flag,normality_pvalue,normality_label,iqr_q1,iqr_q3,iqr_value,iqr_lower_bound,iqr_upper_bound,iqr_lower_count,iqr_upper_count,iqr_total_count,iqr_total_pct,extreme_iqr_count,extreme_iqr_pct,zscore_lower_bound,zscore_upper_bound,zscore_lower_count,zscore_upper_count,zscore_total_count,zscore_total_pct,mod_zscore_mad,mod_zscore_count,mod_zscore_pct,pctl_lower_bound,pctl_upper_bound,pctl_count,pctl_pct,kurtosis,distribution_type_x,is_heavy_tailed,is_skewed,heavy_tail_flag,has_outliers,outlier_severity,suggested_treatment,winsorize_lower,winsorize_upper,distribution_type_y,transformation_recommendation,scaling_needed,clipping_recommendation,winsorization_recommendation,is_numeric,is_discrete,is_near_constant,zero_inflation_flag,binning_opportunity,structure_score,recommended_action,reason,zero_ratio
0,0,AMT_ANNUITY,float64,float64,float64,numeric_continuous,False,False,False,float64,False,False,False,float64,307499,13672,0.044462,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN,float64,12,0.000000,low_missing,mean/median,Mean (if normal) or Median (if skewed),13672,307499,100.000000,"['24700.5', '35698.5', '6750.0']",float64,12.000000,0.000000,"307,499.000000",binary_classification,fisher_exact,0.001851,0.616206,0.203939,False,likely_random_or_weak_signal,1.000000,0.000000,0.080732,0.080732,0.000000,0.203939,lower_positive_rate_when_missin

In [9]:
# Quick report-driven overview
overview = pd.DataFrame({
    "metric": [
        "Total report rows",
        "Potential ID columns",
        "Constant columns",
        "Quasi-constant columns",
        "High-cardinality columns",
        "Very-high-cardinality columns",
        "Skewed numeric columns",
        "Numeric columns with outliers",
        "Binning opportunities",
        "Zero-inflated columns"
    ],
    "value": [
        len(rpt),
        int(rpt["potential_id_flag"].sum()) if "potential_id_flag" in rpt.columns else np.nan,
        int(rpt["constant_flag"].sum()) if "constant_flag" in rpt.columns else np.nan,
        int(rpt["quasi_constant_flag"].sum()) if "quasi_constant_flag" in rpt.columns else np.nan,
        int(rpt["high_cardinality_flag"].sum()) if "high_cardinality_flag" in rpt.columns else np.nan,
        int(rpt["very_high_cardinality_flag"].sum()) if "very_high_cardinality_flag" in rpt.columns else np.nan,
        int(rpt["is_skewed"].sum()) if "is_skewed" in rpt.columns else np.nan,
        int(rpt["has_outliers"].sum()) if "has_outliers" in rpt.columns else np.nan,
        int(rpt["binning_opportunity"].sum()) if "binning_opportunity" in rpt.columns else np.nan,
        int(rpt["zero_inflation_flag"].sum()) if "zero_inflation_flag" in rpt.columns else np.nan
    ]
})
display(overview)

,metric,value
0,Total report rows,584
1,Potential ID columns,1
2,Constant columns,21
3,Quasi-constant columns,157
4,High-cardinality columns,437
5,Very-high-cardinality columns,335
6,Skewed numeric columns,511
7,Numeric columns with outliers,499
8,Binning opportunities,471
9,Zero-inflated columns,250


## 7. Create Working Copies
We will preserve the original train/test dataframes and perform engineering on copies.

In [10]:
train_fe = train_df.copy()
test_fe = test_df.copy()

describe_df(train_fe, "train_fe")
describe_df(test_fe, "test_fe")

train_fe shape: (307511, 584)
train_fe numeric columns: 568
train_fe categorical columns: 16
test_fe shape: (48744, 583)
test_fe numeric columns: 567
test_fe categorical columns: 16


## 8. Report-Driven Drop Candidates
This is a **safe** drop strategy.

### Default drop logic
- Drop the explicit ID column from modeling datasets
- Drop report-flagged potential ID columns
- Drop report-flagged constant columns
- Drop non-structural columns with extremely high missingness

In [11]:
# Structural prefixes where missing often means 'no history'
STRUCTURAL_PREFIXES = ["BUREAU_", "PREV_", "APPROVED_", "REFUSED_", "POS_", "CC_", "INS_"]

structural_cols = family_columns(train_fe, STRUCTURAL_PREFIXES)

potential_id_cols = []
if "potential_id_flag" in rpt.columns:
    potential_id_cols = get_existing_cols(train_fe, rpt.loc[rpt["potential_id_flag"], "column"].tolist())

constant_cols = []
if "constant_flag" in rpt.columns:
    constant_cols = get_existing_cols(train_fe, rpt.loc[rpt["constant_flag"], "column"].tolist())

extreme_missing_non_structural_cols = []
if missing_col in rpt.columns:
    candidates = rpt.loc[rpt[missing_col] >= EXTREME_MISSING_DROP_THRESHOLD, "column"].tolist()
    extreme_missing_non_structural_cols = [c for c in get_existing_cols(train_fe, candidates) if c not in structural_cols]

drop_candidates = sorted(set(
    [ID_COL] * (ID_COL in train_fe.columns) +
    [c for c in potential_id_cols if c not in [TARGET_COL, ID_COL]] +
    [c for c in constant_cols if c not in [TARGET_COL, ID_COL]] +
    [c for c in extreme_missing_non_structural_cols if c not in [TARGET_COL, ID_COL]]
))

print("Potential ID columns from report:", len(potential_id_cols))
print("Constant columns from report    :", len(constant_cols))
print("Extreme missing non-structural  :", len(extreme_missing_non_structural_cols))
print("Final drop candidates           :", len(drop_candidates))
drop_candidates[:50]

Potential ID columns from report: 1
Constant columns from report    : 21
Extreme missing non-structural  : 0
Final drop candidates           : 22


['BUREAU_BB_STATUS_nan_MEAN_MEAN',
 'BUREAU_CREDIT_ACTIVE_nan_MEAN',
 'BUREAU_CREDIT_CURRENCY_nan_MEAN',
 'BUREAU_CREDIT_TYPE_nan_MEAN',
 'CC_NAME_CONTRACT_STATUS_nan_MEAN',
 'POS_NAME_CONTRACT_STATUS_nan_MEAN',
 'PREV_CHANNEL_TYPE_nan_MEAN',
 'PREV_CODE_REJECT_REASON_nan_MEAN',
 'PREV_FLAG_LAST_APPL_PER_CONTRACT_nan_MEAN',
 'PREV_NAME_CASH_LOAN_PURPOSE_nan_MEAN',
 'PREV_NAME_CLIENT_TYPE_nan_MEAN',
 'PREV_NAME_CONTRACT_STATUS_nan_MEAN',
 'PREV_NAME_CONTRACT_TYPE_nan_MEAN',
 'PREV_NAME_GOODS_CATEGORY_House Construction_MEAN',
 'PREV_NAME_GOODS_CATEGORY_nan_MEAN',
 'PREV_NAME_PAYMENT_TYPE_nan_MEAN',
 'PREV_NAME_PORTFOLIO_nan_MEAN',
 'PREV_NAME_PRODUCT_TYPE_nan_MEAN',
 'PREV_NAME_SELLER_INDUSTRY_nan_MEAN',
 'PREV_NAME_YIELD_GROUP_nan_MEAN',
 'PREV_WEEKDAY_APPR_PROCESS_START_nan_MEAN',
 'SK_ID_CURR']

In [12]:
# Apply the safe drops
train_fe.drop(columns=[c for c in drop_candidates if c in train_fe.columns], inplace=True, errors="ignore")
test_fe.drop(columns=[c for c in drop_candidates if c in test_fe.columns], inplace=True, errors="ignore")

describe_df(train_fe, "train_fe after safe drops")
describe_df(test_fe, "test_fe after safe drops")

train_fe after safe drops shape: (307511, 562)
train_fe after safe drops numeric columns: 546
train_fe after safe drops categorical columns: 16
test_fe after safe drops shape: (48744, 561)
test_fe after safe drops numeric columns: 545
test_fe after safe drops categorical columns: 16


## 9. Structural Missing vs True Missing
The structural audit report is used here to decide how missing values should be treated.

### Structural history groups
For families like:
- `BUREAU_*`
- `PREV_*`
- `APPROVED_*`
- `REFUSED_*`
- `POS_*`
- `CC_*`
- `INS_*`

missing often means **this customer has no such history**.

So we will:
1. Create history availability flags
2. Fill structural numeric columns with `0`
3. Fill structural categorical columns with `"Missing_History"`

In [13]:
# Create family-level history availability flags
for prefix in STRUCTURAL_PREFIXES:
    fam_cols = [c for c in train_fe.columns if c.startswith(prefix)]
    if len(fam_cols) == 0:
        continue

    flag_col = f"HAS_{prefix.rstrip('_')}_HISTORY"
    train_fe[flag_col] = (~train_fe[fam_cols].isna().all(axis=1)).astype(int)
    test_fe[flag_col] = (~test_fe[fam_cols].isna().all(axis=1)).astype(int)

    # Optional family coverage ratio
    coverage_col = f"{prefix.rstrip('_')}_NON_NULL_RATIO"
    train_fe[coverage_col] = train_fe[fam_cols].notna().mean(axis=1)
    test_fe[coverage_col] = test_fe[fam_cols].notna().mean(axis=1)

print("Structural history flags created.")

Structural history flags created.


In [14]:
# Fill structural family missing values safely
for prefix in STRUCTURAL_PREFIXES:
    fam_cols = [c for c in train_fe.columns if c.startswith(prefix)]
    if not fam_cols:
        continue

    numeric_fam = train_fe[fam_cols].select_dtypes(include=[np.number]).columns.tolist()
    cat_fam = [c for c in fam_cols if c not in numeric_fam]

    for col in numeric_fam:
        train_fe[col] = train_fe[col].fillna(0)
        test_fe[col] = test_fe[col].fillna(0)

    for col in cat_fam:
        train_fe[col] = train_fe[col].fillna("Missing_History")
        test_fe[col] = test_fe[col].fillna("Missing_History")

print("Structural family missing values filled.")

Structural family missing values filled.


## 10. Missing Indicators and General Imputation
For non-structural features:
- create missing indicators for important missing columns
- numeric -> median imputation
- categorical -> `"Unknown"`

In [15]:
# Create missing indicators for non-structural columns with meaningful missingness
eligible_missing_cols = []
if missing_col in rpt.columns:
    eligible_missing_cols = rpt.loc[
        (rpt[missing_col] >= MISSING_INDICATOR_THRESHOLD) &
        (~rpt["column"].isin(structural_cols)),
        "column"
    ].tolist()

eligible_missing_cols = [c for c in eligible_missing_cols if c in train_fe.columns and c != TARGET_COL]

for col in eligible_missing_cols:
    ind_col = f"{col}_MISSING_FLAG"
    train_fe[ind_col] = train_fe[col].isna().astype(int)
    test_fe[ind_col] = test_fe[col].isna().astype(int)

print("Missing indicators created:", len(eligible_missing_cols))

Missing indicators created: 62


In [16]:
# General numeric and categorical imputation
train_num_cols = train_fe.select_dtypes(include=[np.number]).columns.tolist()
test_num_cols = test_fe.select_dtypes(include=[np.number]).columns.tolist()

# Do not impute target using train medians in the feature list
if TARGET_COL in train_num_cols:
    train_num_cols.remove(TARGET_COL)

cat_cols_train = train_fe.select_dtypes(exclude=[np.number]).columns.tolist()
cat_cols_test = test_fe.select_dtypes(exclude=[np.number]).columns.tolist()

# Median imputation for numeric columns
numeric_fill_map = {}
for col in train_num_cols:
    median_value = train_fe[col].median()
    if pd.isna(median_value):
        median_value = 0
    numeric_fill_map[col] = median_value
    train_fe[col] = train_fe[col].fillna(median_value)
    if col in test_fe.columns:
        test_fe[col] = test_fe[col].fillna(median_value)

# Categorical imputation
for col in cat_cols_train:
    train_fe[col] = train_fe[col].fillna("Unknown")
for col in cat_cols_test:
    test_fe[col] = test_fe[col].fillna("Unknown")

print("General imputation completed.")

General imputation completed.


## 11. Report-Driven Numeric Transformations
This section uses the structural audit report to:
- clip outlier-prone numeric features using report bounds
- create log features for selected skewed positive variables

In [17]:
# Clip numeric features using report-provided winsorization bounds when available
clip_candidates = []
if "clipping_recommendation" in rpt.columns:
    clip_candidates = rpt.loc[rpt["clipping_recommendation"], "column"].tolist()

clip_candidates = [c for c in clip_candidates if c in train_fe.columns and pd.api.types.is_numeric_dtype(train_fe[c]) and c != TARGET_COL]

applied_clip_features = []

for col in clip_candidates:
    lower = None
    upper = None

    if "winsorize_lower" in rpt.columns:
        lower_vals = rpt.loc[rpt["column"] == col, "winsorize_lower"].values
        if len(lower_vals) > 0:
            lower = lower_vals[0]

    if "winsorize_upper" in rpt.columns:
        upper_vals = rpt.loc[rpt["column"] == col, "winsorize_upper"].values
        if len(upper_vals) > 0:
            upper = upper_vals[0]

    train_fe[col] = clip_with_bounds(train_fe[col], lower, upper)
    if col in test_fe.columns:
        test_fe[col] = clip_with_bounds(test_fe[col], lower, upper)

    applied_clip_features.append(col)

print("Clipped features:", len(applied_clip_features))

Clipped features: 264


In [18]:
# Select report-driven log candidates
log_candidates = []
if "transformation_recommendation" in rpt.columns:
    log_candidates = rpt.loc[
        rpt["transformation_recommendation"].astype(str).str.contains("log1p", case=False, na=False),
        "column"
    ].tolist()

# Keep only usable positive numeric columns
usable_log_candidates = []
for col in log_candidates:
    if col in train_fe.columns and col != TARGET_COL and pd.api.types.is_numeric_dtype(train_fe[col]):
        combined_min = min(train_fe[col].min(skipna=True), test_fe[col].min(skipna=True) if col in test_fe.columns else train_fe[col].min(skipna=True))
        if pd.notna(combined_min) and combined_min >= 0:
            usable_log_candidates.append(col)

# Prioritize using structure_score if available
if "structure_score" in rpt.columns:
    score_map = rpt.set_index("column")["structure_score"].to_dict()
    usable_log_candidates = sorted(
        usable_log_candidates,
        key=lambda c: score_map.get(c, 0),
        reverse=True
    )

usable_log_candidates = usable_log_candidates[:MAX_LOG_FEATURES]

for col in usable_log_candidates:
    new_col = f"{col}_LOG1P"
    train_fe[new_col] = np.log1p(train_fe[col].clip(lower=0))
    if col in test_fe.columns:
        test_fe[new_col] = np.log1p(test_fe[col].clip(lower=0))

print("Log-transformed features created:", len(usable_log_candidates))
usable_log_candidates[:20]

Log-transformed features created: 60


['BUREAU_AMT_CREDIT_MAX_OVERDUE_MAX',
 'BUREAU_AMT_CREDIT_MAX_OVERDUE_MEAN',
 'BUREAU_BB_STATUS_1_MEAN_MEAN',
 'BUREAU_BB_STATUS_BAD_FLAG_MEAN_MEAN',
 'BUREAU_BB_STATUS_BAD_FLAG_SUM_MEAN',
 'BUREAU_CREDIT_TYPE_Credit card_MEAN',
 'CC_AMT_BALANCE_MAX',
 'CC_AMT_DRAWINGS_ATM_CURRENT_SUM',
 'CC_AMT_DRAWINGS_CURRENT_MAX',
 'CC_AMT_DRAWINGS_POS_CURRENT_MAX',
 'CC_AMT_DRAWINGS_POS_CURRENT_MEAN',
 'CC_AMT_DRAWINGS_POS_CURRENT_SUM',
 'CC_AMT_INST_MIN_REGULARITY_MAX',
 'CC_AMT_INST_MIN_REGULARITY_MEAN',
 'CC_AMT_PAYMENT_TOTAL_CURRENT_MAX',
 'CC_AMT_PAYMENT_TOTAL_CURRENT_MEAN',
 'CC_AMT_PAYMENT_TOTAL_CURRENT_SUM',
 'CC_AMT_RECEIVABLE_PRINCIPAL_MAX',
 'CC_AMT_TOTAL_RECEIVABLE_MAX',
 'CC_CNT_DRAWINGS_ATM_CURRENT_SUM']

## 12. Domain-Based Feature Engineering
This section creates practical manual features for the Home Credit problem.

In [19]:
# -----------------------------
# A. Time-based human-readable features
# -----------------------------
if "DAYS_BIRTH" in train_fe.columns:
    train_fe["AGE_YEARS"] = np.abs(train_fe["DAYS_BIRTH"]) / 365.25
    test_fe["AGE_YEARS"] = np.abs(test_fe["DAYS_BIRTH"]) / 365.25

if "DAYS_EMPLOYED" in train_fe.columns:
    train_fe["EMPLOYED_YEARS"] = np.abs(train_fe["DAYS_EMPLOYED"]) / 365.25
    test_fe["EMPLOYED_YEARS"] = np.abs(test_fe["DAYS_EMPLOYED"]) / 365.25

if "DAYS_LAST_PHONE_CHANGE" in train_fe.columns:
    train_fe["PHONE_CHANGE_YEARS"] = np.abs(train_fe["DAYS_LAST_PHONE_CHANGE"]) / 365.25
    test_fe["PHONE_CHANGE_YEARS"] = np.abs(test_fe["DAYS_LAST_PHONE_CHANGE"]) / 365.25

if "DAYS_REGISTRATION" in train_fe.columns:
    train_fe["REGISTRATION_YEARS"] = np.abs(train_fe["DAYS_REGISTRATION"]) / 365.25
    test_fe["REGISTRATION_YEARS"] = np.abs(test_fe["DAYS_REGISTRATION"]) / 365.25

if "DAYS_ID_PUBLISH" in train_fe.columns:
    train_fe["ID_PUBLISH_YEARS"] = np.abs(train_fe["DAYS_ID_PUBLISH"]) / 365.25
    test_fe["ID_PUBLISH_YEARS"] = np.abs(test_fe["DAYS_ID_PUBLISH"]) / 365.25

# -----------------------------
# B. Application amount ratios
# -----------------------------
if {"AMT_ANNUITY", "AMT_INCOME_TOTAL"}.issubset(train_fe.columns):
    train_fe["FE_ANNUITY_TO_INCOME"] = safe_divide(train_fe["AMT_ANNUITY"], train_fe["AMT_INCOME_TOTAL"])
    test_fe["FE_ANNUITY_TO_INCOME"] = safe_divide(test_fe["AMT_ANNUITY"], test_fe["AMT_INCOME_TOTAL"])

if {"AMT_CREDIT", "AMT_INCOME_TOTAL"}.issubset(train_fe.columns):
    train_fe["FE_CREDIT_TO_INCOME"] = safe_divide(train_fe["AMT_CREDIT"], train_fe["AMT_INCOME_TOTAL"])
    test_fe["FE_CREDIT_TO_INCOME"] = safe_divide(test_fe["AMT_CREDIT"], test_fe["AMT_INCOME_TOTAL"])

if {"AMT_CREDIT", "AMT_ANNUITY"}.issubset(train_fe.columns):
    train_fe["FE_CREDIT_TO_ANNUITY"] = safe_divide(train_fe["AMT_CREDIT"], train_fe["AMT_ANNUITY"])
    test_fe["FE_CREDIT_TO_ANNUITY"] = safe_divide(test_fe["AMT_CREDIT"], test_fe["AMT_ANNUITY"])

if {"AMT_GOODS_PRICE", "AMT_CREDIT"}.issubset(train_fe.columns):
    train_fe["FE_GOODS_TO_CREDIT"] = safe_divide(train_fe["AMT_GOODS_PRICE"], train_fe["AMT_CREDIT"])
    test_fe["FE_GOODS_TO_CREDIT"] = safe_divide(test_fe["AMT_GOODS_PRICE"], test_fe["AMT_CREDIT"])

if {"AMT_INCOME_TOTAL", "CNT_FAM_MEMBERS"}.issubset(train_fe.columns):
    train_fe["FE_INCOME_PER_PERSON"] = safe_divide(train_fe["AMT_INCOME_TOTAL"], train_fe["CNT_FAM_MEMBERS"])
    test_fe["FE_INCOME_PER_PERSON"] = safe_divide(test_fe["AMT_INCOME_TOTAL"], test_fe["CNT_FAM_MEMBERS"])

if {"AMT_CREDIT", "CNT_FAM_MEMBERS"}.issubset(train_fe.columns):
    train_fe["FE_CREDIT_PER_PERSON"] = safe_divide(train_fe["AMT_CREDIT"], train_fe["CNT_FAM_MEMBERS"])
    test_fe["FE_CREDIT_PER_PERSON"] = safe_divide(test_fe["AMT_CREDIT"], test_fe["CNT_FAM_MEMBERS"])

if {"AMT_INCOME_TOTAL", "CNT_CHILDREN"}.issubset(train_fe.columns):
    train_fe["FE_INCOME_PER_CHILD"] = safe_divide(train_fe["AMT_INCOME_TOTAL"], (train_fe["CNT_CHILDREN"] + 1))
    test_fe["FE_INCOME_PER_CHILD"] = safe_divide(test_fe["AMT_INCOME_TOTAL"], (test_fe["CNT_CHILDREN"] + 1))

# -----------------------------
# C. Time ratios
# -----------------------------
if {"DAYS_EMPLOYED", "DAYS_BIRTH"}.issubset(train_fe.columns):
    train_fe["FE_EMPLOYED_TO_AGE"] = safe_divide(np.abs(train_fe["DAYS_EMPLOYED"]), np.abs(train_fe["DAYS_BIRTH"]))
    test_fe["FE_EMPLOYED_TO_AGE"] = safe_divide(np.abs(test_fe["DAYS_EMPLOYED"]), np.abs(test_fe["DAYS_BIRTH"]))

if {"DAYS_LAST_PHONE_CHANGE", "DAYS_BIRTH"}.issubset(train_fe.columns):
    train_fe["FE_PHONE_TO_AGE"] = safe_divide(np.abs(train_fe["DAYS_LAST_PHONE_CHANGE"]), np.abs(train_fe["DAYS_BIRTH"]))
    test_fe["FE_PHONE_TO_AGE"] = safe_divide(np.abs(test_fe["DAYS_LAST_PHONE_CHANGE"]), np.abs(test_fe["DAYS_BIRTH"]))

if {"DAYS_REGISTRATION", "DAYS_BIRTH"}.issubset(train_fe.columns):
    train_fe["FE_REGISTRATION_TO_AGE"] = safe_divide(np.abs(train_fe["DAYS_REGISTRATION"]), np.abs(train_fe["DAYS_BIRTH"]))
    test_fe["FE_REGISTRATION_TO_AGE"] = safe_divide(np.abs(test_fe["DAYS_REGISTRATION"]), np.abs(test_fe["DAYS_BIRTH"]))

if {"DAYS_ID_PUBLISH", "DAYS_BIRTH"}.issubset(train_fe.columns):
    train_fe["FE_ID_PUBLISH_TO_AGE"] = safe_divide(np.abs(train_fe["DAYS_ID_PUBLISH"]), np.abs(train_fe["DAYS_BIRTH"]))
    test_fe["FE_ID_PUBLISH_TO_AGE"] = safe_divide(np.abs(test_fe["DAYS_ID_PUBLISH"]), np.abs(test_fe["DAYS_BIRTH"]))

# -----------------------------
# D. Social and inquiry summaries
# -----------------------------
social_30 = get_existing_cols(train_fe, ["OBS_30_CNT_SOCIAL_CIRCLE", "DEF_30_CNT_SOCIAL_CIRCLE"])
social_60 = get_existing_cols(train_fe, ["OBS_60_CNT_SOCIAL_CIRCLE", "DEF_60_CNT_SOCIAL_CIRCLE"])
bureau_req_cols = get_existing_cols(train_fe, [
    "AMT_REQ_CREDIT_BUREAU_DAY", "AMT_REQ_CREDIT_BUREAU_HOUR", "AMT_REQ_CREDIT_BUREAU_WEEK",
    "AMT_REQ_CREDIT_BUREAU_MON", "AMT_REQ_CREDIT_BUREAU_QRT", "AMT_REQ_CREDIT_BUREAU_YEAR"
])

if len(social_30) == 2:
    train_fe["FE_SOCIAL_TOTAL_30"] = train_fe[social_30].sum(axis=1)
    test_fe["FE_SOCIAL_TOTAL_30"] = test_fe[social_30].sum(axis=1)

if len(social_60) == 2:
    train_fe["FE_SOCIAL_TOTAL_60"] = train_fe[social_60].sum(axis=1)
    test_fe["FE_SOCIAL_TOTAL_60"] = test_fe[social_60].sum(axis=1)

if {"DEF_30_CNT_SOCIAL_CIRCLE", "OBS_30_CNT_SOCIAL_CIRCLE"}.issubset(train_fe.columns):
    train_fe["FE_DEF_TO_OBS_30"] = safe_divide(train_fe["DEF_30_CNT_SOCIAL_CIRCLE"], train_fe["OBS_30_CNT_SOCIAL_CIRCLE"] + 1)
    test_fe["FE_DEF_TO_OBS_30"] = safe_divide(test_fe["DEF_30_CNT_SOCIAL_CIRCLE"], test_fe["OBS_30_CNT_SOCIAL_CIRCLE"] + 1)

if {"DEF_60_CNT_SOCIAL_CIRCLE", "OBS_60_CNT_SOCIAL_CIRCLE"}.issubset(train_fe.columns):
    train_fe["FE_DEF_TO_OBS_60"] = safe_divide(train_fe["DEF_60_CNT_SOCIAL_CIRCLE"], train_fe["OBS_60_CNT_SOCIAL_CIRCLE"] + 1)
    test_fe["FE_DEF_TO_OBS_60"] = safe_divide(test_fe["DEF_60_CNT_SOCIAL_CIRCLE"], test_fe["OBS_60_CNT_SOCIAL_CIRCLE"] + 1)

if len(bureau_req_cols) > 0:
    train_fe["FE_TOTAL_BUREAU_INQUIRIES"] = train_fe[bureau_req_cols].sum(axis=1)
    test_fe["FE_TOTAL_BUREAU_INQUIRIES"] = test_fe[bureau_req_cols].sum(axis=1)

    train_fe["FE_ANY_BUREAU_INQUIRY"] = (train_fe[bureau_req_cols].sum(axis=1) > 0).astype(int)
    test_fe["FE_ANY_BUREAU_INQUIRY"] = (test_fe[bureau_req_cols].sum(axis=1) > 0).astype(int)

print("Manual domain features created.")

Manual domain features created.


## 13. Housing Trio Consolidation
The structural report suggests many housing columns are repetitive and highly related.

We will summarize common `AVG / MEDI / MODE` groups into:
- mean
- std
- range
- missing count

In [20]:
housing_triplets = {
    "APARTMENTS": ["APARTMENTS_AVG", "APARTMENTS_MEDI", "APARTMENTS_MODE"],
    "BASEMENTAREA": ["BASEMENTAREA_AVG", "BASEMENTAREA_MEDI", "BASEMENTAREA_MODE"],
    "ELEVATORS": ["ELEVATORS_AVG", "ELEVATORS_MEDI", "ELEVATORS_MODE"],
    "ENTRANCES": ["ENTRANCES_AVG", "ENTRANCES_MEDI", "ENTRANCES_MODE"],
    "FLOORSMAX": ["FLOORSMAX_AVG", "FLOORSMAX_MEDI", "FLOORSMAX_MODE"],
    "LIVINGAREA": ["LIVINGAREA_AVG", "LIVINGAREA_MEDI", "LIVINGAREA_MODE"],
    "NONLIVINGAREA": ["NONLIVINGAREA_AVG", "NONLIVINGAREA_MEDI", "NONLIVINGAREA_MODE"],
    "YEARS_BEGINEXPLUATATION": ["YEARS_BEGINEXPLUATATION_AVG", "YEARS_BEGINEXPLUATATION_MEDI", "YEARS_BEGINEXPLUATATION_MODE"],
    "YEARS_BUILD": ["YEARS_BUILD_AVG", "YEARS_BUILD_MEDI", "YEARS_BUILD_MODE"]
}

for group_name, raw_cols in housing_triplets.items():
    cols = get_existing_cols(train_fe, raw_cols)
    if len(cols) < 2:
        continue

    train_fe[f"FE_{group_name}_MEAN"] = train_fe[cols].mean(axis=1)
    test_fe[f"FE_{group_name}_MEAN"] = test_fe[cols].mean(axis=1)

    train_fe[f"FE_{group_name}_STD"] = train_fe[cols].std(axis=1)
    test_fe[f"FE_{group_name}_STD"] = test_fe[cols].std(axis=1)

    train_fe[f"FE_{group_name}_RANGE"] = train_fe[cols].max(axis=1) - train_fe[cols].min(axis=1)
    test_fe[f"FE_{group_name}_RANGE"] = test_fe[cols].max(axis=1) - test_fe[cols].min(axis=1)

    train_fe[f"FE_{group_name}_MISS_COUNT"] = train_fe[cols].isna().sum(axis=1)
    test_fe[f"FE_{group_name}_MISS_COUNT"] = test_fe[cols].isna().sum(axis=1)

# A few useful housing ratios if columns exist
if {"LIVINGAREA_AVG", "TOTALAREA_MODE"}.issubset(train_fe.columns):
    train_fe["FE_LIVING_TO_TOTAL_AREA"] = safe_divide(train_fe["LIVINGAREA_AVG"], train_fe["TOTALAREA_MODE"])
    test_fe["FE_LIVING_TO_TOTAL_AREA"] = safe_divide(test_fe["LIVINGAREA_AVG"], test_fe["TOTALAREA_MODE"])

if {"NONLIVINGAREA_AVG", "TOTALAREA_MODE"}.issubset(train_fe.columns):
    train_fe["FE_NONLIVING_TO_TOTAL_AREA"] = safe_divide(train_fe["NONLIVINGAREA_AVG"], train_fe["TOTALAREA_MODE"])
    test_fe["FE_NONLIVING_TO_TOTAL_AREA"] = safe_divide(test_fe["NONLIVINGAREA_AVG"], test_fe["TOTALAREA_MODE"])

if {"BASEMENTAREA_AVG", "TOTALAREA_MODE"}.issubset(train_fe.columns):
    train_fe["FE_BASEMENT_TO_TOTAL_AREA"] = safe_divide(train_fe["BASEMENTAREA_AVG"], train_fe["TOTALAREA_MODE"])
    test_fe["FE_BASEMENT_TO_TOTAL_AREA"] = safe_divide(test_fe["BASEMENTAREA_AVG"], test_fe["TOTALAREA_MODE"])

print("Housing summary features created.")

Housing summary features created.


## 14. Report-Driven Binning
Only a limited number of bin-friendly features will be binned.

In [21]:
bin_candidates = []
if "binning_opportunity" in rpt.columns:
    bin_candidates = rpt.loc[rpt["binning_opportunity"], "column"].tolist()

# Keep useful numeric columns only
usable_bin_candidates = []
for col in bin_candidates:
    if col in train_fe.columns and col != TARGET_COL and pd.api.types.is_numeric_dtype(train_fe[col]):
        if train_fe[col].nunique(dropna=True) > 8:
            usable_bin_candidates.append(col)

# Prioritize with structure score if available
if "structure_score" in rpt.columns:
    score_map = rpt.set_index("column")["structure_score"].to_dict()
    usable_bin_candidates = sorted(
        usable_bin_candidates,
        key=lambda c: score_map.get(c, 0),
        reverse=True
    )

usable_bin_candidates = usable_bin_candidates[:MAX_BIN_FEATURES]

for col in usable_bin_candidates:
    try:
        train_fe[f"{col}_BIN_Q5"] = pd.qcut(train_fe[col], q=5, duplicates="drop").astype(str)
        # Reuse train bins on combined category labels for test using rank approach fallback
        # To keep the notebook robust, bin test separately with qcut if possible.
        try:
            test_fe[f"{col}_BIN_Q5"] = pd.qcut(test_fe[col], q=5, duplicates="drop").astype(str)
        except Exception:
            test_fe[f"{col}_BIN_Q5"] = pd.cut(test_fe[col], bins=5).astype(str)
    except Exception:
        pass

print("Binned features created from report:", len(usable_bin_candidates))
usable_bin_candidates[:20]

Binned features created from report: 20


['BUREAU_AMT_CREDIT_MAX_OVERDUE_MAX',
 'BUREAU_AMT_CREDIT_MAX_OVERDUE_MEAN',
 'BUREAU_BB_MONTHS_BALANCE_MAX_MEAN',
 'BUREAU_BB_STATUS_1_MEAN_MEAN',
 'BUREAU_BB_STATUS_BAD_FLAG_MEAN_MEAN',
 'BUREAU_BB_STATUS_BAD_FLAG_SUM_MEAN',
 'BUREAU_CREDIT_TYPE_Credit card_MEAN',
 'CC_AMT_BALANCE_MAX',
 'CC_AMT_BALANCE_MEAN',
 'CC_AMT_BALANCE_SUM',
 'CC_AMT_DRAWINGS_ATM_CURRENT_SUM',
 'CC_AMT_DRAWINGS_CURRENT_MAX',
 'CC_AMT_DRAWINGS_CURRENT_MEAN',
 'CC_AMT_DRAWINGS_CURRENT_SUM',
 'CC_AMT_DRAWINGS_POS_CURRENT_MAX',
 'CC_AMT_DRAWINGS_POS_CURRENT_MEAN',
 'CC_AMT_DRAWINGS_POS_CURRENT_SUM',
 'CC_AMT_INST_MIN_REGULARITY_MAX',
 'CC_AMT_INST_MIN_REGULARITY_MEAN',
 'CC_AMT_PAYMENT_TOTAL_CURRENT_MAX']

## 15. Categorical Encoding Strategy

### Strategy
- Low-cardinality categorical columns -> one-hot encoding
- High-cardinality categorical columns -> frequency encoding using train frequencies
- Keep the process train-test consistent

In [22]:
# Recompute categorical columns after feature creation
train_cat_cols = train_fe.select_dtypes(exclude=[np.number]).columns.tolist()
test_cat_cols = test_fe.select_dtypes(exclude=[np.number]).columns.tolist()

# Build high-cardinality candidates from report
report_high_card_cols = []
if "high_cardinality_flag" in rpt.columns:
    report_high_card_cols = rpt.loc[rpt["high_cardinality_flag"], "column"].tolist()

high_card_cols = [c for c in train_cat_cols if (c in report_high_card_cols) or (train_fe[c].nunique(dropna=False) > HIGH_CARDINALITY_THRESHOLD)]
low_card_cols = [c for c in train_cat_cols if c not in high_card_cols and train_fe[c].nunique(dropna=False) <= LOW_CARDINALITY_THRESHOLD]

print("Categorical columns after feature creation:", len(train_cat_cols))
print("High-cardinality categorical columns      :", len(high_card_cols))
print("Low-cardinality categorical columns       :", len(low_card_cols))

Categorical columns after feature creation: 36
High-cardinality categorical columns      : 1
Low-cardinality categorical columns       : 34


In [23]:
# Frequency encoding for high-cardinality categorical columns
frequency_encoding_maps = {}

for col in high_card_cols:
    freq_map = train_fe[col].value_counts(normalize=True, dropna=False).to_dict()
    frequency_encoding_maps[col] = freq_map

    train_fe[f"{col}_FREQ_ENC"] = train_fe[col].map(freq_map).fillna(0)
    if col in test_fe.columns:
        test_fe[f"{col}_FREQ_ENC"] = test_fe[col].map(freq_map).fillna(0)

print("Frequency encoding completed.")

Frequency encoding completed.


In [24]:
# One-hot encode low-cardinality categorical columns
# Use concatenation to guarantee aligned dummy columns
train_index = train_fe.index
test_index = test_fe.index

combined_for_dummies = pd.concat(
    [
        train_fe[low_card_cols].assign(__dataset__="train"),
        test_fe[low_card_cols].assign(__dataset__="test")
    ],
    axis=0
)

combined_dummies = pd.get_dummies(combined_for_dummies, columns=low_card_cols, dummy_na=False, drop_first=False)
train_dummy = combined_dummies[combined_dummies["__dataset__"] == "train"].drop(columns="__dataset__")
test_dummy = combined_dummies[combined_dummies["__dataset__"] == "test"].drop(columns="__dataset__")

train_dummy.index = train_index
test_dummy.index = test_index

# Drop original encoded columns and attach dummies
train_fe.drop(columns=low_card_cols, inplace=True, errors="ignore")
test_fe.drop(columns=low_card_cols, inplace=True, errors="ignore")

train_fe = pd.concat([train_fe, train_dummy], axis=1)
test_fe = pd.concat([test_fe, test_dummy], axis=1)

print("One-hot encoding completed.")

One-hot encoding completed.


In [25]:
# Drop original high-cardinality categorical columns after frequency encoding
train_fe.drop(columns=high_card_cols, inplace=True, errors="ignore")
test_fe.drop(columns=high_card_cols, inplace=True, errors="ignore")

# Align train and test columns
train_target = None
if TARGET_COL in train_fe.columns:
    train_target = train_fe[TARGET_COL].copy()
    train_fe_no_target = train_fe.drop(columns=[TARGET_COL])
else:
    train_fe_no_target = train_fe.copy()

train_fe_no_target, test_fe = ensure_same_columns(train_fe_no_target, test_fe)

# Bring target back
if train_target is not None:
    train_fe = pd.concat([train_fe_no_target, train_target], axis=1)
else:
    train_fe = train_fe_no_target.copy()

print("Train and test alignment completed.")
describe_df(train_fe, "train_fe final before save")
describe_df(test_fe, "test_fe final before save")

Train and test alignment completed.
train_fe final before save shape: (307511, 875)
train_fe final before save numeric columns: 744
train_fe final before save categorical columns: 131
test_fe final before save shape: (48744, 874)
test_fe final before save numeric columns: 743
test_fe final before save categorical columns: 131


## 16. Final Cleanup
Replace remaining infinite values and perform a final null check.

In [26]:
# Replace infinities with NaN, then fill numeric NaNs with 0 as a final safeguard
for df_name, df_obj in [("train_fe", train_fe), ("test_fe", test_fe)]:
    df_obj.replace([np.inf, -np.inf], np.nan, inplace=True)

    num_cols = df_obj.select_dtypes(include=[np.number]).columns.tolist()
    if df_name == "train_fe" and TARGET_COL in num_cols:
        num_cols.remove(TARGET_COL)

    for col in num_cols:
        df_obj[col] = df_obj[col].fillna(0)

    cat_cols = df_obj.select_dtypes(exclude=[np.number]).columns.tolist()
    for col in cat_cols:
        df_obj[col] = df_obj[col].fillna("Unknown")

print("Final null-safety cleanup completed.")
print("Remaining nulls in train:", int(train_fe.isna().sum().sum()))
print("Remaining nulls in test :", int(test_fe.isna().sum().sum()))

Final null-safety cleanup completed.
Remaining nulls in train: 0
Remaining nulls in test : 0


## 17. Engineering Summary Tables

In [27]:
# Build a high-level engineering summary
feature_summary = pd.DataFrame({
    "metric": [
        "Original train rows",
        "Original train columns",
        "Original test rows",
        "Original test columns",
        "Final train rows",
        "Final train columns",
        "Final test rows",
        "Final test columns",
        "Dropped columns",
        "Structural history flags created",
        "Missing indicators created",
        "Clipped features",
        "Log features created",
        "Binned features attempted",
        "High-card frequency encoded",
        "Low-card one-hot encoded"
    ],
    "value": [
        train_df.shape[0],
        train_df.shape[1],
        test_df.shape[0],
        test_df.shape[1],
        train_fe.shape[0],
        train_fe.shape[1],
        test_fe.shape[0],
        test_fe.shape[1],
        len(drop_candidates),
        len([c for c in train_fe.columns if c.startswith("HAS_") and c.endswith("_HISTORY")]),
        len(eligible_missing_cols),
        len(applied_clip_features),
        len(usable_log_candidates),
        len(usable_bin_candidates),
        len(high_card_cols),
        len(low_card_cols)
    ]
})

display(feature_summary)

,metric,value
0,Original train rows,307511
1,Original train columns,584
2,Original test rows,48744
3,Original test columns,583
4,Final train rows,307511
5,Final train columns,875
6,Final test rows,48744
7,Final test columns,874
8,Dropped columns,22
9,Structural history flags created,7


In [28]:
# Show a compact sample of newly created engineered columns
new_feature_prefixes = ("FE_", "HAS_", "PREV_NON_NULL_RATIO", "BUREAU_NON_NULL_RATIO", "APPROVED_NON_NULL_RATIO",
                        "REFUSED_NON_NULL_RATIO", "POS_NON_NULL_RATIO", "CC_NON_NULL_RATIO", "INS_NON_NULL_RATIO")
new_feature_cols = [c for c in train_fe.columns if c.startswith(new_feature_prefixes) or c.endswith("_LOG1P") or c.endswith("_BIN_Q5") or c.endswith("_MISSING_FLAG")]
print("Number of engineered columns detected:", len(new_feature_cols))
new_feature_cols[:100]

Number of engineered columns detected: 192


['HAS_BUREAU_HISTORY',
 'BUREAU_NON_NULL_RATIO',
 'HAS_PREV_HISTORY',
 'PREV_NON_NULL_RATIO',
 'HAS_APPROVED_HISTORY',
 'APPROVED_NON_NULL_RATIO',
 'HAS_REFUSED_HISTORY',
 'REFUSED_NON_NULL_RATIO',
 'HAS_POS_HISTORY',
 'POS_NON_NULL_RATIO',
 'HAS_CC_HISTORY',
 'CC_NON_NULL_RATIO',
 'HAS_INS_HISTORY',
 'INS_NON_NULL_RATIO',
 'AMT_REQ_CREDIT_BUREAU_DAY_MISSING_FLAG',
 'AMT_REQ_CREDIT_BUREAU_HOUR_MISSING_FLAG',
 'AMT_REQ_CREDIT_BUREAU_MON_MISSING_FLAG',
 'AMT_REQ_CREDIT_BUREAU_QRT_MISSING_FLAG',
 'AMT_REQ_CREDIT_BUREAU_WEEK_MISSING_FLAG',
 'AMT_REQ_CREDIT_BUREAU_YEAR_MISSING_FLAG',
 'APARTMENTS_AVG_MISSING_FLAG',
 'APARTMENTS_MEDI_MISSING_FLAG',
 'APARTMENTS_MODE_MISSING_FLAG',
 'APP_CAR_BIRTH_RATIO_MISSING_FLAG',
 'APP_EMPLOYED_BIRTH_RATIO_MISSING_FLAG',
 'APP_EXT_SOURCE_STD_MISSING_FLAG',
 'APP_PHONE_EMPLOYED_RATIO_MISSING_FLAG',
 'BASEMENTAREA_AVG_MISSING_FLAG',
 'BASEMENTAREA_MEDI_MISSING_FLAG',
 'BASEMENTAREA_MODE_MISSING_FLAG',
 'COMMONAREA_AVG_MISSING_FLAG',
 'COMMONAREA_MEDI_MISSI

## 18. Save Final Outputs to Processed_2

In [29]:
# Output file paths
TRAIN_SAVE_PATH = os.path.join(SAVE_DIR, "final_train_after_feature_engineering.csv")
TEST_SAVE_PATH = os.path.join(SAVE_DIR, "final_test_after_feature_engineering.csv")
SUMMARY_SAVE_PATH = os.path.join(SAVE_DIR, "feature_engineering_summary.csv")
METADATA_SAVE_PATH = os.path.join(SAVE_DIR, "feature_engineering_metadata.json")

# Save CSV outputs
train_fe.to_csv(TRAIN_SAVE_PATH, index=False)
test_fe.to_csv(TEST_SAVE_PATH, index=False)
feature_summary.to_csv(SUMMARY_SAVE_PATH, index=False)

metadata = {
    "target_column": TARGET_COL,
    "id_column": ID_COL,
    "original_train_shape": list(train_df.shape),
    "original_test_shape": list(test_df.shape),
    "final_train_shape": list(train_fe.shape),
    "final_test_shape": list(test_fe.shape),
    "drop_candidates": drop_candidates,
    "structural_prefixes": STRUCTURAL_PREFIXES,
    "missing_indicator_columns": eligible_missing_cols,
    "clipped_features": applied_clip_features,
    "log_features_source_columns": usable_log_candidates,
    "bin_candidates_source_columns": usable_bin_candidates,
    "high_cardinality_categorical_columns": high_card_cols,
    "low_cardinality_categorical_columns": low_card_cols,
    "engineered_feature_count_detected": len(new_feature_cols)
}

with open(METADATA_SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print("Saved successfully.")
print("Train file   ->", TRAIN_SAVE_PATH)
print("Test file    ->", TEST_SAVE_PATH)
print("Summary file ->", SUMMARY_SAVE_PATH)
print("Metadata     ->", METADATA_SAVE_PATH)

Saved successfully.
Train file   -> C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Data\Processed_2\final_train_after_feature_engineering.csv
Test file    -> C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Data\Processed_2\final_test_after_feature_engineering.csv
Summary file -> C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Data\Processed_2\feature_engineering_summary.csv
Metadata     -> C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Data\Processed_2\feature_engineering_metadata.json


## 19. Final Recommendation Notes

### What this notebook already does
- Uses the structural audit report as the main engineering guide
- Distinguishes structural missing from true missing
- Creates practical manual features
- Applies report-driven clipping and log transforms
- Encodes categorical columns safely
- Saves final outputs into **Processed_2**

### What you can do next
- Build a baseline model notebook using these engineered files
- Run feature importance with LightGBM / XGBoost
- Create a feature selection notebook after baseline modeling